# Modelo de Control de Inventarios — ABC-XYZ

Pipeline completo de planificación de inventarios para ~3000 SKUs:

1. **Carga de datos** — reportes transaccionales, artículos (stock actual), consolidado (metadata logística)
2. **Clasificación ABC** — por valor acumulado de ingresos (AA / A / B / C)
3. **Clasificación XYZ** — por coeficiente de variación de demanda semanal (X / Y / Z)
4. **Matriz ABC-XYZ** — asigna nivel de servicio dinámico por celda
5. **Modelo de inventario v2** — ROP empírico/paramétrico + EOQ en múltiplos de caja
6. **Resumen y análisis** — distribución de pedidos, inversión, CBM

In [103]:
import pandas as pd
import numpy as np
import math
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print("Librerías cargadas ✓")

Librerías cargadas ✓


---
## 1. Carga de Datos

In [104]:
# ══════════════════════════════════════════════════════════════════════════════
# CARGA DE REPORTES TRANSACCIONALES
# ══════════════════════════════════════════════════════════════════════════════
reports = [i + 4 for i in range(12)]

df_temp = [
    pd.read_csv(f"reporte ({r}).csv", encoding="latin1", low_memory=False)
    for r in reports
]
df = pd.concat(df_temp, ignore_index=True)

print(f"Reportes cargados: {len(reports)}")
print(f"Transacciones totales: {len(df):,} filas")
print(f"Columnas: {list(df.columns)}")
df.head(3)

Reportes cargados: 12
Transacciones totales: 181,778 filas
Columnas: ['Centro de Operaciones Empresa', 'Condición de Venta', 'Categoría Cliente Padre', 'Categoría Cliente', 'Categoría Artículo Padre', 'Categoría Artículo', 'Código Cliente', 'Nombre Cliente', 'Código Centro Operaciones Cliente', 'Nombre Centro Operaciones Cliente', 'Agente de Venta', 'Punto Venta', 'Fabricante', 'Marca', 'Modelo', 'Código Artículo', 'Nombre Artículo', 'Tipo Transacción', 'Código Transacción', '¿Factura Sistema Anterior?', 'Código Transacción Referencia', 'Fecha Transacción', 'Unidad Medida', 'Cantidad', 'Unidad Medida Peso', 'Peso', 'Divisa', 'Precio Unitario', 'Subtotal Bruto', 'Descuento', 'Subtotal Neto', 'Impuesto', 'Total', 'Costo Unitario', 'Total Costo', 'Observación']


,Centro de Operaciones Empresa,Condición de Venta,Categoría Cliente Padre,Categoría Cliente,Categoría Artículo Padre,Categoría Artículo,Código Cliente,Nombre Cliente,Código Centro Operaciones Cliente,Nombre Centro Operaciones Cliente,Agente de Venta,Punto Venta,Fabricante,Marca,Modelo,...,Fecha Transacción,Unidad Medida,Cantidad,Unidad Medida Peso,Peso,Divisa,Precio Unitario,Subtotal Bruto,Descuento,Subtotal Neto,Impuesto,Total,Costo Unitario,Total Costo,Observación
0,006-Sede PGR 2023,CR-Crédito,-,CLIE-02-CLIENTES JURIDICOS,PRO-PRODUCTOS,PRO-05-ESTETICA Y BELLEZA,130764,LIZ ANDREINA SANCHEZ VEGAS TORTOZA,130764,OFICINA PRINCIPAL,ALEJANDOR RODRIGUEZ,CAJA 14,-,-,-,...,01/08/2026 04:50 PM,Unid,1.0000,-,-,$,35.0000,35.00,0.0000,35.00,0.0000,35.00,6.4500,6.45,SP-196016522\nINICIAL 0$
1,006-Sede PGR 2023,CR-Crédito,-,CLIE-02-CLIENTES JURIDICOS,PRO-PRODUCTOS,PRO-08-ADULTOS,103041,RONALD MIJAIL SALAZAR CADENAS,103041,OFICINA PRINCIPAL,ALEJANDOR RODRIGUEZ,CAJA 14,-,-,-,...,01/08/2026 04:57 PM,Unid,1.0000,-,-,$,25.0000,25.00,0.0000,25.00,0.0000,25.00,5.1900,5.19,SP-196017185\nINICIAL 0$
2,006-Sede PGR 2023,CR-Crédito,-,CLIE-02-CLIENTES JURIDICOS,PRO-PRODUCTOS,PRO-24-REVESTIMIENTO,115653,EVER ALEXANDER NAVA DUGARTE,115653,OFICINA PRINCIPAL,ALEJANDOR RODRIGUEZ,CAJA 14,-,-,-,...,01/08/2026 05:02 PM,Unid,1.0000,-,-,$,25.0000,25.00,0.0000,25.00,0.0000,25.00,8.6000,8.60,SP-196019940\nINICIAL 0$


In [105]:
# ══════════════════════════════════════════════════════════════════════════════
# CARGA DE ARTÍCULOS (STOCK ACTUAL)
# ══════════════════════════════════════════════════════════════════════════════
df_articulos = pd.read_csv('articulos.csv', encoding='latin1')
df_articulos = df_articulos.loc[:, ['Código Artículo', 'Existencia Período']]

# Limpieza del stock
df_articulos['Existencia Período'] = (
    df_articulos['Existencia Período']
    .astype(str).str.strip().str.replace(',', '', regex=False)
)
df_articulos['Existencia Período'] = pd.to_numeric(
    df_articulos['Existencia Período'], errors='coerce'
).fillna(0.0)

print(f"Artículos cargados: {len(df_articulos):,}")
df_articulos.head()

Artículos cargados: 1,126


,Código Artículo,Existencia Período
0,MASS0018,237.0000
1,MASS0020,256.0000
2,MASS0021,1.0000
3,MASS0023,124.0000
4,MASS0025,1.0000


In [106]:
# ══════════════════════════════════════════════════════════════════════════════
# CARGA DE CONSOLIDADO (METADATA LOGÍSTICA)
# ══════════════════════════════════════════════════════════════════════════════
df_consolidado = pd.read_csv("consolidado.csv", encoding="latin1")
df_consolidado = df_consolidado.loc[
    :, ['contenedor', 'codigo', 'cantidad', 'costo', 'cantidad_por_caja', 'CBMM']
]
df_consolidado.dropna(subset=['codigo', 'CBMM', 'cantidad_por_caja'], inplace=True)
df_consolidado['costo'] = pd.to_numeric(
    df_consolidado['costo'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip(),
    errors='coerce'
)


# Expandir códigos multi-línea (algunos campos tienen varios códigos separados por \n)
df_expandido = (
    df_consolidado.assign(
        codigo=(
            df_consolidado['codigo']
            .astype(str)
            .str.strip()
            .str.split(r'\\n|[\n\s]+')
        )
    )
    .explode('codigo')
    .dropna(subset=['codigo'])
)
df_expandido['codigo'] = df_expandido['codigo'].str.strip()
df_expandido = df_expandido[df_expandido['codigo'] != ''].reset_index(drop=True)

print(f"Consolidado original: {len(df_consolidado):,} filas")
print(f"Consolidado expandido: {len(df_expandido):,} filas")
df_expandido.head()

Consolidado original: 1,818 filas
Consolidado expandido: 2,681 filas


,contenedor,codigo,cantidad,costo,cantidad_por_caja,CBMM
0,CONTENEDORES JULIO,MASS2118,400.0000,1.0000,400.0000,0.0524
1,CONTENEDORES JULIO,MASS0409,450.0000,4.2000,30.0000,0.0866
2,CONTENEDORES JULIO,MASS2068,240.0000,7.2500,30.0000,0.0959
3,CONTENEDORES JULIO,MASS0927,240.0000,1.2000,60.0000,0.0272
4,CONTENEDORES JULIO,MASS1452,240.0000,1.2500,24.0000,0.0256


In [122]:
# ══════════════════════════════════════════════════════════════════════════════
# PREPARACIÓN DE DATOS TRANSACCIONALES
# ══════════════════════════════════════════════════════════════════════════════
_colss = ['Código Artículo', 'Fecha Transacción', 'Cantidad', 'Total', 'Total Costo', 'Categoría Artículo']
df_new = df.loc[:, _colss].copy()

# Limpieza de Código Artículo
df_new['Código Artículo'] = df_new['Código Artículo'].astype(str).str.strip()

# Limpieza de Total (ingresos)
df_new['Total'] = (
    df_new['Total'].astype(str).str.strip().str.replace(',', '', regex=False)
)
df_new['Total'] = pd.to_numeric(df_new['Total'], errors='coerce')

# Limpieza de Total Costo (FIX: usar 'Total Costo' como fuente, NO 'Total')
df_new['Total Costo'] = (
    df_new['Total Costo'].astype(str).str.strip().str.replace(',', '', regex=False)
)
df_new['Total Costo'] = pd.to_numeric(df_new['Total Costo'], errors='coerce')

# Limpieza de Cantidad
df_new['Cantidad'] = pd.to_numeric(df_new['Cantidad'], errors='coerce')

print(f"Datos transaccionales: {len(df_new):,} filas")
print(f"SKUs únicos: {df_new['Código Artículo'].nunique():,}")
df_new.head()

Datos transaccionales: 181,778 filas
SKUs únicos: 2,048


,Código Artículo,Fecha Transacción,Cantidad,Total,Total Costo,Categoría Artículo
0,MASS2651,01/08/2026 04:50 PM,1.0000,35.0000,6.4500,PRO-05-ESTETICA Y BELLEZA
1,MASS0287,01/08/2026 04:57 PM,1.0000,25.0000,5.1900,PRO-08-ADULTOS
2,MASS1842,01/08/2026 05:02 PM,1.0000,25.0000,8.6000,PRO-24-REVESTIMIENTO
3,MASS2328,01/08/2026 05:06 PM,1.0000,360.0000,115.1500,PRO-04-DEPORTE Y FITNESS
4,MASS0364,01/08/2026 05:15 PM,1.0000,18.0000,4.3900,PRO-08-ADULTOS


In [108]:
# ══════════════════════════════════════════════════════════════════════════════
# MERGE DE METADATA (consolidado + artículos → transacciones)
# ══════════════════════════════════════════════════════════════════════════════

# Maestro de metadata logística (un registro por código, el último disponible)
maestro = df_expandido[['codigo', 'costo', 'cantidad_por_caja', 'CBMM']].drop_duplicates(
    subset=['codigo'], keep='last'
)

# Left join: transacciones ← metadata logística
df_resultado = pd.merge(
    df_new, maestro,
    how='left',
    left_on='Código Artículo',
    right_on='codigo',
).drop(columns=['codigo'])

# Left join: + stock actual de artículos
df_merged = pd.merge(
    df_resultado, df_articulos,
    how='left',
    on='Código Artículo',
)

# Limpieza de Existencia Período
df_merged['Existencia Período'] = (
    df_merged['Existencia Período']
    .astype(str).str.strip().str.replace(',', '', regex=False)
)
df_merged['Existencia Período'] = pd.to_numeric(
    df_merged['Existencia Período'], errors='coerce'
).fillna(0.0)

print(f"Dataset combinado: {len(df_merged):,} filas × {len(df_merged.columns)} columnas")
print(f"Columnas: {list(df_merged.columns)}")
df_merged['Fecha Transacción'] = pd.to_datetime(df_merged['Fecha Transacción'], format='%d/%m/%Y %I:%M %p')

Dataset combinado: 199,803 filas × 9 columnas
Columnas: ['Código Artículo', 'Fecha Transacción', 'Cantidad', 'Total', 'Total Costo', 'costo', 'cantidad_por_caja', 'CBMM', 'Existencia Período']


In [109]:
# 1. Encontrar la fecha más reciente en los reportes
fecha_maxima = df_merged['Fecha Transacción'].max()

# 2. Calcular la fecha límite (6 meses hacia atrás)
fecha_corte_6m = fecha_maxima - pd.DateOffset(months=6)

print(f"Filtrando transacciones desde: {fecha_corte_6m.date()} hasta: {fecha_maxima.date()}")


df_ultimos_6m = df_merged[df_merged['Fecha Transacción'] >= fecha_corte_6m].copy()
df_merged = df_ultimos_6m
df_merged

Filtrando transacciones desde: 2026-02-23 hasta: 2026-08-23


,Código Artículo,Fecha Transacción,Cantidad,Total,Total Costo,costo,cantidad_por_caja,CBMM,Existencia Período
0,MASS2651,2026-08-01 16:50:00,1.0000,35.0000,6.4500,7.2500,30.0000,0.0959,31.0000
1,MASS0287,2026-08-01 16:57:00,1.0000,25.0000,5.1900,5.0000,40.0000,0.0119,156.0000
2,MASS1842,2026-08-01 17:02:00,1.0000,25.0000,8.6000,5.0000,10.0000,0.1750,66.0000
3,MASS2328,2026-08-01 17:06:00,1.0000,360.0000,115.1500,62.0000,1.0000,0.0961,0.0000
4,MASS0364,2026-08-01 17:15:00,1.0000,18.0000,4.3900,3.0300,50.0000,0.0245,5.0000
...,...,...,...,...,...,...,...,...,...
70944,MASS0696,2026-02-28 12:45:00,1.0000,3.4900,0.3000,0.2000,100.0000,0.0187,0.0000
70945,MASS0518,2026-02-28 12:45:00,2.0000,14.0000,1.7600,0.8600,300.0000,0.0580,1.0000
70946,MASS0518,2026-02-28 12:45:00,2.0000,14.0000,1.7600,0.8600,300.0000,0.0580,1.0000
70947,MASS0902,2026-02-28 12:47:00,1.0000,26.9800,7.1000,3.5000,20.0000,0.1267,1.0000


---
## 2. Clasificación ABC-XYZ

### Lógica:
- **ABC** → por valor acumulado de ingresos (Principio de Pareto)
  - AA: ≤ 50% acumulado  
  - A: ≤ 80% acumulado  
  - B: ≤ 95% acumulado  
  - C: > 95% acumulado
  
- **XYZ** → por coeficiente de variación (CV) de la demanda semanal
  - X: CV ≤ 0.5 (demanda estable, fácil de pronosticar)  
  - Y: 0.5 < CV ≤ 1.0 (demanda variable, con patrones)  
  - Z: CV > 1.0 (demanda errática/intermitente)

In [110]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASIFICACIÓN ABC POR INGRESOS (VALOR ACUMULADO REAL)
# ══════════════════════════════════════════════════════════════════════════════

ventas_totales = df_merged['Total'].sum()

ventas_agrupadas = (
    df_merged.groupby('Código Artículo')
    .agg(total_ventas=('Total', 'sum'))
    .sort_values(by='total_ventas', ascending=False)
)

ventas_agrupadas['Porcentaje'] = ventas_agrupadas['total_ventas'] / ventas_totales
ventas_agrupadas['Porcentaje_acum'] = ventas_agrupadas['Porcentaje'].cumsum()

condiciones_abc = [
    ventas_agrupadas['Porcentaje_acum'] <= 0.50,   # AA: top 50% del valor
    ventas_agrupadas['Porcentaje_acum'] <= 0.80,   # A:  50-80%
    ventas_agrupadas['Porcentaje_acum'] <= 0.95,   # B:  80-95%
    ventas_agrupadas['Porcentaje_acum'] > 0.95,    # C:  95-100%
]
valores_abc = ['AA', 'A', 'B', 'C']
ventas_agrupadas['clase_abc'] = np.select(condiciones_abc, valores_abc, default='C')

df_abc = ventas_agrupadas[['total_ventas', 'Porcentaje', 'Porcentaje_acum', 'clase_abc']].copy()

print("═" * 60)
print("DISTRIBUCIÓN ABC POR INGRESOS")
print("═" * 60)
for clase in ['AA', 'A', 'B', 'C']:
    subset = df_abc[df_abc['clase_abc'] == clase]
    print(f"  {clase:>2}: {len(subset):>5} SKUs | "
          f"${subset['total_ventas'].sum():>12,.2f} | "
          f"{subset['Porcentaje'].sum() * 100:>5.1f}% del total")
print(f"{'─' * 60}")
print(f"  Total: {len(df_abc):>4} SKUs | ${ventas_totales:>12,.2f}")
print()
df_abc.head(10)

════════════════════════════════════════════════════════════
DISTRIBUCIÓN ABC POR INGRESOS
════════════════════════════════════════════════════════════
  AA:    72 SKUs | $  826,943.71 |  49.8% del total
   A:   247 SKUs | $  501,838.49 |  30.2% del total
   B:   417 SKUs | $  249,338.75 |  15.0% del total
   C:   779 SKUs | $   83,083.71 |   5.0% del total
────────────────────────────────────────────────────────────
  Total: 1515 SKUs | $1,661,204.66



,total_ventas,Porcentaje,Porcentaje_acum,clase_abc
Código Artículo,,,,
MASS3063,"110,807.6400",0.0667,0.0667,AA
MASS2603,"48,000.1000",0.0289,0.0956,AA
MASS1575,"33,695.2200",0.0203,0.1159,AA
MASS2304,"22,471.3100",0.0135,0.1294,AA
MASS2303,"20,575.7000",0.0124,0.1418,AA
MASS0518,"19,982.4400",0.0120,0.1538,AA
MASS1039,"19,063.5500",0.0115,0.1653,AA
MASS1806,"17,853.5100",0.0107,0.1760,AA
MASS1377,"17,677.1200",0.0106,0.1867,AA


In [111]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASIFICACIÓN XYZ POR COEFICIENTE DE VARIACIÓN (CV)
# ══════════════════════════════════════════════════════════════════════════════

# Parsear fechas para la serie temporal
df_fechas = df_merged.dropna(subset=['Fecha Transacción', 'Cantidad']).copy()
df_fechas['Fecha Transacción'] = pd.to_datetime(
    df_fechas['Fecha Transacción'], format='mixed', dayfirst=True
)

# Calcular CV de demanda semanal por SKU
cv_results = []
for sku, grupo in df_fechas.groupby('Código Artículo'):
    serie_semanal = (
        grupo.set_index('Fecha Transacción')
        .resample('W-MON')['Cantidad']
        .sum()
        .fillna(0.0)
    )
    mean_d = float(serie_semanal.mean())
    std_d = float(serie_semanal.std(ddof=1)) if len(serie_semanal) > 1 else 0.0
    cv = std_d / mean_d if mean_d > 0 else float('inf')
    
    cv_results.append({
        'Código Artículo': sku,
        'demanda_sem_prom': mean_d,
        'demanda_sem_std': std_d,
        'cv': cv,
        'n_semanas': len(serie_semanal),
    })

df_xyz = pd.DataFrame(cv_results).set_index('Código Artículo')

# Clasificar XYZ
condiciones_xyz = [
    df_xyz['cv'] <= 0.5,
    df_xyz['cv'] <= 1.0,
    df_xyz['cv'] > 1.0,
]
valores_xyz = ['X', 'Y', 'Z']
df_xyz['clase_xyz'] = np.select(condiciones_xyz, valores_xyz, default='Z')

print("═" * 60)
print("DISTRIBUCIÓN XYZ POR VARIABILIDAD DE DEMANDA")
print("═" * 60)
for clase in ['X', 'Y', 'Z']:
    subset = df_xyz[df_xyz['clase_xyz'] == clase]
    cv_median = subset['cv'].replace([np.inf], np.nan).median()
    print(f"  {clase}: {len(subset):>5} SKUs | CV mediana: {cv_median:>6.2f}")
print()
df_xyz.head(10)

════════════════════════════════════════════════════════════
DISTRIBUCIÓN XYZ POR VARIABILIDAD DE DEMANDA
════════════════════════════════════════════════════════════
  X:   261 SKUs | CV mediana:   0.00
  Y:   202 SKUs | CV mediana:   0.84
  Z:  1052 SKUs | CV mediana:   1.71



,demanda_sem_prom,demanda_sem_std,cv,n_semanas,clase_xyz
Código Artículo,,,,,
MASS0003,1.0000,0.0000,0.0000,1,X
MASS0018,14.5000,13.3292,0.9193,4,Y
MASS0020,2.3846,6.3754,2.6736,26,Z
MASS0021,26.2609,23.8351,0.9076,23,Y
MASS0023,2.6800,4.6163,1.7225,25,Z
MASS0025,0.0625,4.2657,68.2505,16,Z
MASS0040,38.1818,34.1351,0.8940,22,Y
MASS0046,2.4000,3.4244,1.4268,20,Z
MASS0047,1.7692,2.8887,1.6327,26,Z


In [112]:
# ══════════════════════════════════════════════════════════════════════════════
# MATRIZ ABC-XYZ COMBINADA + NIVELES DE SERVICIO DINÁMICOS
# ══════════════════════════════════════════════════════════════════════════════

# Combinar ABC y XYZ
df_abc_xyz = df_abc[['clase_abc']].join(df_xyz[['cv', 'clase_xyz']], how='outer')

# SKUs sin datos ABC → clase C (conservador)
df_abc_xyz['clase_abc'] = df_abc_xyz['clase_abc'].fillna('C')
# SKUs sin datos XYZ → clase Z (conservador)
df_abc_xyz['clase_xyz'] = df_abc_xyz['clase_xyz'].fillna('Z')
# Clase combinada
df_abc_xyz['clase_abc_xyz'] = df_abc_xyz['clase_abc'] + '-' + df_abc_xyz['clase_xyz']

# ┌──────────────────────────────────────────────────────────────┐
# │ NIVELES DE SERVICIO POR CELDA ABC-XYZ                       │
# │                                                              │
# │ Filas = valor del SKU (ABC)                                  │
# │ Columnas = estabilidad de demanda (XYZ)                      │
# │                                                              │
# │         X (estable)   Y (variable)   Z (errático)            │
# │ AA       0.98           0.97           0.95                  │
# │ A        0.95           0.93           0.90                  │
# │ B        0.90           0.88           0.85                  │
# │ C        0.85           0.80           0.75                  │
# └──────────────────────────────────────────────────────────────┘

NIVEL_SERVICIO_ABC_XYZ = {
    ('AA', 'X'): 0.98,  ('AA', 'Y'): 0.97,  ('AA', 'Z'): 0.95,
    ('A',  'X'): 0.95,  ('A',  'Y'): 0.93,  ('A',  'Z'): 0.90,
    ('B',  'X'): 0.90,  ('B',  'Y'): 0.88,  ('B',  'Z'): 0.85,
    ('C',  'X'): 0.85,  ('C',  'Y'): 0.80,  ('C',  'Z'): 0.75,
}

df_abc_xyz['nivel_servicio'] = df_abc_xyz.apply(
    lambda row: NIVEL_SERVICIO_ABC_XYZ.get(
        (row['clase_abc'], row['clase_xyz']), 0.85
    ), axis=1
)

# Distribución de la matriz
print("═" * 60)
print("MATRIZ ABC-XYZ (conteo de SKUs)")
print("═" * 60)
matriz = pd.crosstab(
    df_abc_xyz['clase_abc'], df_abc_xyz['clase_xyz'],
    margins=True, margins_name='Total'
)
# Reordenar filas y columnas
orden_abc = [c for c in ['AA', 'A', 'B', 'C', 'Total'] if c in matriz.index]
orden_xyz = [c for c in ['X', 'Y', 'Z', 'Total'] if c in matriz.columns]
print(matriz.loc[orden_abc, orden_xyz])

print()
print("NIVELES DE SERVICIO ASIGNADOS:")
print("─" * 60)
for abc in ['AA', 'A', 'B', 'C']:
    row_parts = []
    for xyz in ['X', 'Y', 'Z']:
        ns = NIVEL_SERVICIO_ABC_XYZ.get((abc, xyz), 0.85)
        row_parts.append(f"{xyz}={ns:.0%}")
    print(f"  {abc:>2}: {' | '.join(row_parts)}")

df_abc_xyz.head(10)

════════════════════════════════════════════════════════════
MATRIZ ABC-XYZ (conteo de SKUs)
════════════════════════════════════════════════════════════
clase_xyz    X    Y     Z  Total
clase_abc                       
AA           1   19    52     72
A            8   55   184    247
B           45   70   302    417
C          207   58   514    779
Total      261  202  1052   1515

NIVELES DE SERVICIO ASIGNADOS:
────────────────────────────────────────────────────────────
  AA: X=98% | Y=97% | Z=95%
   A: X=95% | Y=93% | Z=90%
   B: X=90% | Y=88% | Z=85%
   C: X=85% | Y=80% | Z=75%


,clase_abc,cv,clase_xyz,clase_abc_xyz,nivel_servicio
Código Artículo,,,,,
MASS0003,C,0.0000,X,C-X,0.8500
MASS0018,C,0.9193,Y,C-Y,0.8000
MASS0020,C,2.6736,Z,C-Z,0.7500
MASS0021,A,0.9076,Y,A-Y,0.9300
MASS0023,C,1.7225,Z,C-Z,0.7500
MASS0025,C,68.2505,Z,C-Z,0.7500
MASS0040,A,0.8940,Y,A-Y,0.9300
MASS0046,C,1.4268,Z,C-Z,0.7500
MASS0047,C,1.6327,Z,C-Z,0.7500


---
## 3. Modelo de Inventario v2

### Mejoras implementadas:
- **Stock actual** → último registro por fecha (no el primero)
- **Metadata** → separada de transacciones, sin perder filas de demanda por datos faltantes
- **ROP empírico** → solo con ≥3 ciclos de LT no solapados; fallback a paramétrico Normal
- **Nivel de servicio** → dinámico por clasificación ABC-XYZ (no fijo para todos)
- **Etiquetado** → cada SKU indica método ROP, clase ABC-XYZ y nivel de servicio aplicado

In [113]:
# ══════════════════════════════════════════════════════════════════════════════
# FUNCIÓN DE CÁLCULO DE INVENTARIO v2
# ══════════════════════════════════════════════════════════════════════════════

def calcular_estadisticas_inventario(
    df_ventas: pd.DataFrame,
    costo_unitario_fob: float,
    cbm_por_caja: float,
    flete_cbm: float,
    capacidad_contenedor_cbm: float,
    tasa_mantenimiento: float,
    costo_orden_admin: float,
    lead_time_semanas: int,
    unidades_por_caja: int,
    col_cantidad: str = 'Cantidad',
    nivel_servicio: float = 0.90,
    stock_actual: float = 0.0,
    en_transito: float = 0.0,
    backorders: float = 0.0,
    capping_quantile: float = 0.95,
    min_ciclos_para_empirico: int = 3,
) -> dict:
    """
    Calcula métricas de inventario para un SKU individual.
    
    v2: ROP empírico con bloques independientes (no rolling solapado)
        + fallback paramétrico Normal cuando historial insuficiente.
    """
    # 1. COSTOS ─────────────────────────────────────────────────────────────
    cbm_unitario = cbm_por_caja / unidades_por_caja
    costo_flete_unitario = cbm_unitario * flete_cbm
    costo_puesto = costo_unitario_fob + costo_flete_unitario

    # 2. LIMPIEZA DE OUTLIERS ───────────────────────────────────────────────
    ventas_positivas = df_ventas[df_ventas[col_cantidad] > 0][col_cantidad]

    if not ventas_positivas.empty:
        cap_demanda = float(ventas_positivas.quantile(capping_quantile))
        cantidad_clean = df_ventas[col_cantidad].clip(lower=0.0, upper=cap_demanda)
    else:
        cantidad_clean = df_ventas[col_cantidad].clip(lower=0.0)

    demanda_semanal_prom = float(cantidad_clean.mean())
    demanda_semanal_std = float(cantidad_clean.std(ddof=1)) if len(cantidad_clean) > 1 else 0.0
    demanda_anual = demanda_semanal_prom * 52.0
    demanda_esperada_lt = demanda_semanal_prom * lead_time_semanas

    # 3. ROP: EMPÍRICO CON BLOQUES INDEPENDIENTES ──────────────────────────
    n_semanas = len(cantidad_clean)
    n_ciclos_disponibles = n_semanas // lead_time_semanas

    if n_ciclos_disponibles >= min_ciclos_para_empirico:
        # Suficientes ciclos → percentil empírico sobre bloques no solapados
        valores = cantidad_clean.values[: n_ciclos_disponibles * lead_time_semanas]
        demanda_lt_bloques = valores.reshape(n_ciclos_disponibles, lead_time_semanas).sum(axis=1)
        rop = float(np.percentile(demanda_lt_bloques, nivel_servicio * 100))
        metodo_rop = 'empirico'
    else:
        # Historial corto → fallback paramétrico Normal
        z = float(stats.norm.ppf(nivel_servicio))
        std_lt = demanda_semanal_std * math.sqrt(lead_time_semanas)
        rop = demanda_esperada_lt + z * std_lt
        metodo_rop = 'parametrico_normal'

    ss = max(0.0, rop - demanda_esperada_lt)

    # 4. EOQ + MÚLTIPLOS DE CAJA ───────────────────────────────────────────
    h = costo_puesto * tasa_mantenimiento
    eoq_teorico = (
        math.sqrt((2 * demanda_anual * costo_orden_admin) / h) if h > 0 else 0.0
    )
    cajas_a_pedir = (
        math.ceil(eoq_teorico / unidades_por_caja) if unidades_por_caja > 0 else 0
    )
    unidades_a_pedir = cajas_a_pedir * unidades_por_caja
    cbm_total_pedido = cajas_a_pedir * cbm_por_caja
    porcentaje_contenedor = (cbm_total_pedido / capacidad_contenedor_cbm) * 100
    inversion_fob = unidades_a_pedir * costo_unitario_fob

    # 5. DECISIÓN DE COMPRA ────────────────────────────────────────────────
    posicion_inventario = float(stock_actual) + float(en_transito) - float(backorders)
    requiere_pedido = bool(posicion_inventario <= rop)
    margen_disponible = max(0.0, posicion_inventario - rop) if not requiere_pedido else 0.0

    return {
        'demanda_semanal_prom': demanda_semanal_prom,
        'demanda_semanal_std': demanda_semanal_std,
        'demanda_esperada_lt': demanda_esperada_lt,
        'stock_seguridad': ss,
        'rop': rop,
        'metodo_rop': metodo_rop,
        'n_ciclos_disponibles': n_ciclos_disponibles,
        'posicion_inventario': posicion_inventario,
        'requiere_pedido': requiere_pedido,
        'estado': 'REORDENAR' if requiere_pedido else 'OK',
        'unidades_a_pedir': unidades_a_pedir if requiere_pedido else 0,
        'cajas_a_pedir': cajas_a_pedir if requiere_pedido else 0,
        'cbm_total_pedido': cbm_total_pedido if requiere_pedido else 0.0,
        'porcentaje_contenedor': porcentaje_contenedor if requiere_pedido else 0.0,
        'inversion_fob_usd': inversion_fob if requiere_pedido else 0.0,
        'margen_disponible': margen_disponible,
        'eoq_teorico': eoq_teorico,
        'cbm_unitario': cbm_unitario,
        'costo_puesto_unitario': costo_puesto,
        'costo_mantenimiento_h': h,
    }

print("Función calcular_estadisticas_inventario definida ✓")

Función calcular_estadisticas_inventario definida ✓


In [116]:
# ══════════════════════════════════════════════════════════════════════════════
# PLANIFICACIÓN DE INVENTARIO — MOTOR VECTORIZADO ABC-XYZ (SIN BUCLES LENTOS)
# ══════════════════════════════════════════════════════════════════════════════

PARAMETROS_LOGISTICOS = {
    'flete_cbm': 500,
    'capacidad_contenedor_cbm': 68.0,
    'tasa_mantenimiento': 0.15,
    'costo_orden_admin': 50.0,
    'lead_time_semanas': 15,
}

LT = PARAMETROS_LOGISTICOS['lead_time_semanas']

# 1. Asegurar tipos numéricos y fechas
df_valido = df_merged.dropna(subset=['Cantidad', 'Fecha Transacción']).copy()
df_valido['Fecha Transacción'] = pd.to_datetime(df_valido['Fecha Transacción'], format='mixed', dayfirst=True)
df_valido['Cantidad'] = pd.to_numeric(df_valido['Cantidad'], errors='coerce').fillna(0.0)

# 2. Matriz semanal continua de demanda (Todos los SKUs en 1 sola operación)
df_semanal = df_valido.groupby(['Código Artículo', pd.Grouper(key='Fecha Transacción', freq='W-MON')])['Cantidad'].sum().unstack(fill_value=0.0)

# 3. Maestro de Metadata limpia por SKU (con fallback si falta costo en consolidado)
df_sku_meta = df_merged.groupby('Código Artículo').agg({
    'costo': lambda x: float(x.dropna().iloc[-1]) if not x.dropna().empty else np.nan,
    'CBMM': lambda x: float(x.dropna().iloc[-1]) if not x.dropna().empty else 0.05,
    'cantidad_por_caja': lambda x: int(x.dropna().iloc[-1]) if not x.dropna().empty and x.dropna().iloc[-1] > 0 else 1,
    'Existencia Período': lambda x: float(x.dropna().iloc[-1]) if not x.dropna().empty else 0.0,
    'Total Costo': lambda x: float(pd.to_numeric(x, errors='coerce').dropna().mean()) if not x.dropna().empty else 1.0,
    'Total': lambda x: float(pd.to_numeric(x, errors='coerce').dropna().sum()) if not x.dropna().empty else 0.0
})
df_sku_meta['costo'] = df_sku_meta['costo'].fillna(df_sku_meta['Total Costo']).replace(0.0, np.nan).fillna(1.0)

# 4. Estadísticas de Demanda
demanda_media = df_semanal.mean(axis=1)
demanda_desv = df_semanal.std(axis=1, ddof=1).fillna(0.0)
cv = np.where(demanda_media > 0, demanda_desv / demanda_media, 0.0)

# 5. Clasificación ABC (Pareto por Ingresos)
total_ventas = df_sku_meta['Total']
ventas_ordenadas = total_ventas.sort_values(ascending=False)
pct_acum = (ventas_ordenadas / ventas_ordenadas.sum()).cumsum()
clase_abc = pd.Series('C', index=ventas_ordenadas.index)
clase_abc[pct_acum <= 0.95] = 'B'
clase_abc[pct_acum <= 0.80] = 'A'
clase_abc[pct_acum <= 0.50] = 'AA'

# 6. Clasificación XYZ (Volatilidad CV)
clase_xyz = pd.Series('Z', index=df_semanal.index)
clase_xyz[cv <= 1.0] = 'Y'
clase_xyz[cv <= 0.5] = 'X'

# 7. Matriz de Nivel de Servicio Dinámico
NIVEL_SERVICIO_MAP = {
    ('AA', 'X'): 0.98, ('AA', 'Y'): 0.97, ('AA', 'Z'): 0.95,
    ('A',  'X'): 0.95, ('A',  'Y'): 0.93, ('A',  'Z'): 0.90,
    ('B',  'X'): 0.90, ('B',  'Y'): 0.88, ('B',  'Z'): 0.85,
    ('C',  'X'): 0.85, ('C',  'Y'): 0.80, ('C',  'Z'): 0.75,
}

# 8. Construcción de Tabla de Planificación
df_planificacion = pd.DataFrame(index=df_semanal.index)
df_planificacion['demanda_semanal_prom'] = demanda_media
df_planificacion['demanda_semanal_std'] = demanda_desv
df_planificacion['cv'] = cv
df_planificacion['clase_abc'] = clase_abc.reindex(df_planificacion.index).fillna('C')
df_planificacion['clase_xyz'] = clase_xyz.reindex(df_planificacion.index).fillna('Z')
df_planificacion['clase_abc_xyz'] = df_planificacion['clase_abc'] + '-' + df_planificacion['clase_xyz']
df_planificacion['nivel_servicio'] = [NIVEL_SERVICIO_MAP.get((a, x), 0.85) for a, x in zip(df_planificacion['clase_abc'], df_planificacion['clase_xyz'])]

# Unir metadata y asegurar que no contenga NaNs/ceros en divisores
df_planificacion = df_planificacion.join(df_sku_meta[['costo', 'CBMM', 'cantidad_por_caja', 'Existencia Período']])
df_planificacion['costo'] = df_planificacion['costo'].fillna(1.0).clip(lower=0.01)
df_planificacion['CBMM'] = df_planificacion['CBMM'].fillna(0.05).clip(lower=0.001)
df_planificacion['cantidad_por_caja'] = pd.to_numeric(df_planificacion['cantidad_por_caja'], errors='coerce').fillna(1).clip(lower=1).astype(int)
df_planificacion['stock_actual'] = df_planificacion['Existencia Período'].fillna(0.0)

# 9. Cálculos de Inventario: ROP y EOQ en cajas
df_planificacion['cbm_unitario'] = df_planificacion['CBMM'] / df_planificacion['cantidad_por_caja']
df_planificacion['costo_puesto'] = df_planificacion['costo'] + (df_planificacion['cbm_unitario'] * PARAMETROS_LOGISTICOS['flete_cbm'])
df_planificacion['demanda_esperada_lt'] = df_planificacion['demanda_semanal_prom'] * LT

z_scores = stats.norm.ppf(df_planificacion['nivel_servicio'])
df_planificacion['stock_seguridad'] = z_scores * df_planificacion['demanda_semanal_std'] * math.sqrt(LT)
df_planificacion['rop'] = df_planificacion['demanda_esperada_lt'] + df_planificacion['stock_seguridad']

# EOQ protegido contra NaNs e Infinitos
h = df_planificacion['costo_puesto'] * PARAMETROS_LOGISTICOS['tasa_mantenimiento']
demanda_anual = df_planificacion['demanda_semanal_prom'] * 52.0
eoq_unidades = np.where(h > 0, np.sqrt((2 * demanda_anual * PARAMETROS_LOGISTICOS['costo_orden_admin']) / h), 0.0)
eoq_unidades = np.nan_to_num(eoq_unidades, nan=0.0, posinf=0.0, neginf=0.0)

# 10. Decisión de Compra protegida contra IntCastingNaNError
df_planificacion['requiere_pedido'] = df_planificacion['stock_actual'] <= df_planificacion['rop']
df_planificacion['estado'] = np.where(df_planificacion['requiere_pedido'], 'REORDENAR', 'OK')

cajas_calculadas = np.ceil(eoq_unidades / df_planificacion['cantidad_por_caja'])
cajas_seguras = pd.Series(cajas_calculadas, index=df_planificacion.index).replace([np.inf, -np.inf], 0).fillna(0).astype(int)

df_planificacion['cajas_a_pedir'] = np.where(df_planificacion['requiere_pedido'], cajas_seguras, 0)
df_planificacion['unidades_a_pedir'] = df_planificacion['cajas_a_pedir'] * df_planificacion['cantidad_por_caja']
df_planificacion['cbm_total_pedido'] = df_planificacion['cajas_a_pedir'] * df_planificacion['CBMM']
df_planificacion['inversion_fob_usd'] = df_planificacion['unidades_a_pedir'] * df_planificacion['costo']

# Ordenar por volumen CBM
df_planificacion = df_planificacion.sort_values(by='cbm_total_pedido', ascending=False)

# Diagnóstico de salida
requieren = df_planificacion[df_planificacion['requiere_pedido'] == True]
print("=" * 60)
print("PLANIFICACIÓN COMPLETADA CON ÉXITO")
print("=" * 60)
print(f"  SKUs totales analizados:   {len(df_planificacion):>6,}")
print(f"  SKUs a pedir (reorden):   {len(requieren):>6,} ({len(requieren)/len(df_planificacion)*100:.1f}%)")
print(f"  Inversión FOB estimada:   ${requieren['inversion_fob_usd'].sum():>12,.2f}")
print(f"  CBM total a pedir:         {requieren['cbm_total_pedido'].sum():>12,.2f} CBM")
print(f"  Contenedores de 68 CBM:    {requieren['cbm_total_pedido'].sum() / PARAMETROS_LOGISTICOS['capacidad_contenedor_cbm']:>12,.1f}")

df_planificacion.head(10)


PLANIFICACIÓN COMPLETADA CON ÉXITO
  SKUs totales analizados:    1,515
  SKUs a pedir (reorden):      964 (63.6%)
  Inversión FOB estimada:   $  804,901.42
  CBM total a pedir:               782.22 CBM
  Contenedores de 68 CBM:            11.5


,demanda_semanal_prom,demanda_semanal_std,cv,clase_abc,clase_xyz,clase_abc_xyz,nivel_servicio,costo,CBMM,cantidad_por_caja,Existencia Período,stock_actual,cbm_unitario,costo_puesto,demanda_esperada_lt,stock_seguridad,rop,requiere_pedido,estado,cajas_a_pedir,unidades_a_pedir,cbm_total_pedido,inversion_fob_usd
Código Artículo,,,,,,,,,,,,,,,,,,,,,,,
MASS3063,"1,720.9231","3,519.9470",2.0454,AA,Z,AA-Z,0.9500,116.9740,0.0500,1,0.0000,0.0000,0.0500,141.9740,"25,813.8462","22,423.7896","48,237.6358",True,REORDENAR,649,649,32.4500,"75,916.1548"
MASS0722,83.9231,27.3495,0.3259,A,X,A-X,0.9500,1.0000,0.0500,1,0.0000,0.0000,0.0500,26.0000,"1,258.8462",174.2296,"1,433.0757",True,REORDENAR,335,335,16.7500,335.0000
MASS2294,29.9231,28.7359,0.9603,AA,Y,AA-Y,0.9700,4.2608,0.0500,1,1.0000,1.0000,0.0500,29.2608,448.8462,209.3206,658.1668,True,REORDENAR,189,189,9.4500,805.2867
MASS1575,219.5769,255.8497,1.1652,AA,Z,AA-Z,0.9500,214.2978,0.0500,1,"3,900.0000","3,900.0000",0.0500,239.2978,"3,293.6538","1,629.8879","4,923.5418",True,REORDENAR,179,179,8.9500,"38,359.3022"
MASS0021,23.2308,23.9404,1.0305,A,Z,A-Z,0.9000,0.5985,0.0500,1,1.0000,1.0000,0.0500,25.5985,348.4615,118.8266,467.2882,True,REORDENAR,178,178,8.9000,106.5388
MASS2304,14.3077,17.1062,1.1956,AA,Z,AA-Z,0.9500,8.0000,0.0663,1,0.0000,0.0000,0.0663,41.1250,214.6154,108.9748,323.5902,True,REORDENAR,110,110,7.2875,880.0000
MASS2603,49.7692,116.2887,2.3366,AA,Z,AA-Z,0.9500,72.9059,0.0500,1,"1,310.0000","1,310.0000",0.0500,97.9059,746.5385,740.8162,"1,487.3546",True,REORDENAR,133,133,6.6500,"9,696.4810"
MASS2303,11.0385,15.1855,1.3757,AA,Z,AA-Z,0.9500,9.0000,0.0663,1,148.0000,148.0000,0.0663,42.1250,165.5769,96.7389,262.3158,True,REORDENAR,96,96,6.3600,864.0000
MASS2497,11.8846,26.0589,2.1927,A,Z,A-Z,0.9000,1.6215,0.0500,1,0.0000,0.0000,0.0500,26.6215,178.2692,129.3415,307.6107,True,REORDENAR,125,125,6.2500,202.6856


In [121]:
df_planificacion[df_planificacion['clase_abc_xyz'] == 'A-X']

,demanda_semanal_prom,demanda_semanal_std,cv,clase_abc,clase_xyz,clase_abc_xyz,nivel_servicio,costo,CBMM,cantidad_por_caja,Existencia Período,stock_actual,cbm_unitario,costo_puesto,demanda_esperada_lt,stock_seguridad,rop,requiere_pedido,estado,cajas_a_pedir,unidades_a_pedir,cbm_total_pedido,inversion_fob_usd
Código Artículo,,,,,,,,,,,,,,,,,,,,,,,
MASS0722,83.9231,27.3495,0.3259,A,X,A-X,0.9500,1.0000,0.0500,1,0.0000,0.0000,0.0500,26.0000,"1,258.8462",174.2296,"1,433.0757",True,REORDENAR,335,335,16.7500,335.0000


---
## 4. Resultados y Análisis

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# RESUMEN DE RESULTADOS
# ══════════════════════════════════════════════════════════════════════════════

requieren = df_planificacion[df_planificacion['requiere_pedido'] == True]
no_requieren = df_planificacion[df_planificacion['requiere_pedido'] == False]

print("═" * 70)
print("RESUMEN DE PLANIFICACIÓN DE INVENTARIO")
print("═" * 70)
print(f"  SKUs analizados:        {len(df_planificacion):>6}")
print(f"  Requieren pedido:       {len(requieren):>6}  ({len(requieren)/len(df_planificacion)*100:.1f}%)")
print(f"  No requieren pedido:    {len(no_requieren):>6}  ({len(no_requieren)/len(df_planificacion)*100:.1f}%)")
print()
print(f"  Inversión FOB total:    ${requieren['inversion_fob_usd'].sum():>12,.2f}")
print(f"  CBM total pedido:       {requieren['cbm_total_pedido'].sum():>12,.2f} CBM")
print(f"  Contenedores (68 CBM):  {requieren['cbm_total_pedido'].sum() / 68:>12,.1f}")
print()

# Distribución por clase ABC-XYZ
print("─" * 70)
print("PEDIDOS POR CLASE ABC-XYZ:")
print("─" * 70)
resumen_clase = (
    requieren.groupby('clase_abc_xyz')
    .agg(
        n_skus=('requiere_pedido', 'count'),
        inversion_usd=('inversion_fob_usd', 'sum'),
        cbm_total=('cbm_total_pedido', 'sum'),
    )
    .sort_values('inversion_usd', ascending=False)
)
resumen_clase['pct_inversion'] = resumen_clase['inversion_usd'] / resumen_clase['inversion_usd'].sum() * 100
print(resumen_clase.to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# VISTA DETALLADA: SKUs QUE REQUIEREN PEDIDO
# ══════════════════════════════════════════════════════════════════════════════

# Lista de columnas deseadas
cols_deseadas = [
    'clase_abc_xyz', 'nivel_servicio', 'demanda_semanal_prom',
    'rop', 'stock_seguridad', 'stock_actual', 'estado',
    'cajas_a_pedir', 'unidades_a_pedir', 'cbm_total_pedido', 'inversion_fob_usd'
]

# Filtrar solo las columnas que realmente existen en el DataFrame (evita KeyErrors)

print("Top 20 SKUs con mayor pedido (por CBM):")
df_planificacion.loc[df_planificacion['requiere_pedido'] == True].head(20)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ANÁLISIS DE LA MATRIZ ABC-XYZ EN df_planificacion
# ══════════════════════════════════════════════════════════════════════════════

print("═" * 70)
print("DISTRIBUCIÓN EN df_planificacion POR CELDA ABC-XYZ")
print("═" * 70)

# Conteo
print("\nConteo de SKUs:")
print(pd.crosstab(
    df_planificacion['clase_abc'], df_planificacion['clase_xyz'],
    margins=True, margins_name='Total'
).loc[
    [c for c in ['AA', 'A', 'B', 'C', 'Total'] if c in df_planificacion['clase_abc'].unique() or c == 'Total'],
    [c for c in ['X', 'Y', 'Z', 'Total'] if c in df_planificacion['clase_xyz'].unique() or c == 'Total']
])

# % que requieren pedido por celda
print("\n% que requiere pedido por celda:")
pct_pedido = (
    df_planificacion.groupby(['clase_abc', 'clase_xyz'])['requiere_pedido']
    .mean()
    .unstack(fill_value=0)
    * 100
)
print(pct_pedido.round(1))

# Inversión por celda
print("\nInversión FOB (USD) por celda:")
inv_celda = (
    df_planificacion[df_planificacion['requiere_pedido'] == True]
    .groupby(['clase_abc', 'clase_xyz'])['inversion_fob_usd']
    .sum()
    .unstack(fill_value=0)
)
print(inv_celda.round(2))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# EXPORTAR RESULTADOS
# ══════════════════════════════════════════════════════════════════════════════

# Exportar planificación completa
df_planificacion.to_csv('planificacion_abc_xyz.csv', encoding='utf-8-sig')
print("Exportado: planificacion_abc_xyz.csv")

# Exportar solo los que requieren pedido
requieren_export = df_planificacion[df_planificacion['requiere_pedido'] == True]
requieren_export.to_csv('pedidos_requeridos.csv', encoding='utf-8-sig')
print(f"Exportado: pedidos_requeridos.csv ({len(requieren_export)} SKUs)")

# Exportar clasificación ABC-XYZ completa
df_abc_xyz.to_csv('clasificacion_abc_xyz.csv', encoding='utf-8-sig')
print(f"Exportado: clasificacion_abc_xyz.csv ({len(df_abc_xyz)} SKUs)")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# INFO DEL DATASET FINAL
# ══════════════════════════════════════════════════════════════════════════════
df_planificacion.info()